In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import itertools
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
import seaborn as sns
from tqdm import tqdm

from src.analysis import analogy
from src.analysis.state_space import StateSpaceAnalysisSpec, \
    prepare_state_trajectory, flatten_trajectory
from src.datasets.speech_equivalence import SpeechHiddenStateDataset


In [ ]:
base_model = "w2v2_8"

model_class = "discrim-rnn_32-mAP1"
model_name = "word_broad_10frames_fixedlen25"

inflection_results_path = "outputs/analogy/inputs/librispeech-train-clean-100/w2v2/inflection_results.parquet"
# all_cross_instances_path = "all_cross_instances.parquet"
all_cross_instances_path = "outputs/analogy/inputs/librispeech-train-clean-100/w2v2/all_cross_instances.parquet"
most_common_allomorphs_path = "outputs/analogy/inputs/librispeech-train-clean-100/w2v2/most_common_allomorphs.csv"
false_friends_path = "outputs/analogy/inputs/librispeech-train-clean-100/w2v2/false_friends.csv"

train_dataset = "librispeech-train-clean-100"
# hidden_states_path = f"outputs/hidden_states/{base_model}/{train_dataset}.h5"
hidden_states_path = f"/scratch/jgauthier/{base_model}_{train_dataset}.h5"
state_space_specs_path = f"outputs/analogy/inputs/librispeech-train-clean-100/w2v2/state_space_spec.h5"
embeddings_path = f"outputs/model_embeddings/{train_dataset}/{base_model}/{model_class}/{model_name}/{train_dataset}.npy"

output_dir = f"."

pos_counts_path = "data/pos_counts.pkl"

seed = 42

metric = "cosine"

agg_fns = [
    "mean",
]

## Prepare model representations

In [ ]:
if embeddings_path == "ID":
    model_representations = SpeechHiddenStateDataset.from_hdf5(hidden_states_path).states
else:
    with open(embeddings_path, "rb") as f:
        model_representations: np.ndarray = np.load(f)
state_space_spec = StateSpaceAnalysisSpec.from_hdf5(state_space_specs_path)
assert state_space_spec.is_compatible_with(model_representations)

In [ ]:
trajectory_agg = prepare_state_trajectory(model_representations, state_space_spec, 
                                          agg_fn_spec="mean", agg_fn_dimension=1)

In [ ]:
agg, agg_src = flatten_trajectory(trajectory_agg)

## Prepare metadata

In [ ]:
# cuts_df = state_space_spec.cuts.xs("phoneme", level="level").drop(columns=["onset_frame_idx", "offset_frame_idx"])
# cuts_df["label_idx"] = cuts_df.index.get_level_values("label").map({l: i for i, l in enumerate(state_space_spec.labels)})
# cuts_df["frame_idx"] = cuts_df.groupby(["label", "instance_idx"]).cumcount()
# cuts_df = cuts_df.reset_index().set_index(["label", "instance_idx", "frame_idx"]).sort_index()

In [ ]:
# cut_phonemic_forms = cuts_df.groupby(["label", "instance_idx"]).description.agg(' '.join)

In [ ]:
# word_freq_df = pd.read_csv("data/WorldLex_Eng_US.Freq.2.txt", sep="\t", index_col="Word")
# # compute weighted average frequency across domains
# word_freq_df["BlogFreq_rel"] = word_freq_df.BlogFreq / word_freq_df.BlogFreq.sum()
# word_freq_df["TwitterFreq_rel"] = word_freq_df.TwitterFreq / word_freq_df.TwitterFreq.sum()
# word_freq_df["NewsFreq_rel"] = word_freq_df.NewsFreq / word_freq_df.NewsFreq.sum()
# word_freq_df["Freq"] = word_freq_df[["BlogFreq", "TwitterFreq", "NewsFreq"]].mean(axis=1) \
#     * word_freq_df[["BlogFreq", "TwitterFreq", "NewsFreq"]].sum().mean()
# word_freq_df["LogFreq"] = np.log10(word_freq_df.Freq)

In [ ]:
all_cross_instances = pd.read_parquet(all_cross_instances_path)

In [ ]:
inflection_results_df = pd.read_parquet(inflection_results_path)

In [ ]:
most_common_allomorphs = pd.read_csv(most_common_allomorphs_path)
false_friends_df = pd.read_csv(false_friends_path)

## Homophone preparation

In [ ]:
# from collections import defaultdict


# pron2label = defaultdict(set)
# for label, rows in cut_phonemic_forms.groupby("label"):
#     for pron in set(rows):
#         pron2label[pron].add(label)

# homophone_map = defaultdict(set)
# for label_idx, label in enumerate(state_space_spec.labels):
#     for pron in set(cut_phonemic_forms.loc[label]):
#         homophone_map[label] |= pron2label[pron]

In [ ]:
# # Prepare to exclude predictions of homophones from analogy evaluations.
# # create a map from label idx -> all label idxs which should be ignored.
# homophone_map = {state_space_spec.labels.index(label): {state_space_spec.labels.index(hom) for hom in homs}
#                  for label, homs in homophone_map.items()}

## Behavioral tests

In [ ]:
import torch
from src.analysis.analogy import nxn_cos_sim

In [ ]:
device = "cuda"
agg_gpu = torch.tensor(agg).to(device)
agg_src_gpu = torch.tensor(agg_src).to(device)

In [ ]:
flat_idx_lookup = {(label_idx, instance_idx): flat_idx
                       for flat_idx, (label_idx, instance_idx, _) in enumerate(agg_src)}

In [ ]:
def run_same_word_trial(base: str, inflected: str, inflection: str, max_num_vector_samples=50):
    base_idx = state_space_spec.labels.index(base)
    inflected_idx = state_space_spec.labels.index(inflected)

    test_instances = all_cross_instances[
        ((all_cross_instances.base == base) & (all_cross_instances.inflected == inflected)) |
        ((all_cross_instances.base == inflected) & (all_cross_instances.inflected == base))
    ]

    # get all instances of each word
    base_flat_idxs = [flat_idx_lookup[(base_idx, inst)] for inst in test_instances.base_instance_idx.unique()]
    inflected_flat_idxs = [flat_idx_lookup[(inflected_idx, inst)] for inst in test_instances.inflected_instance_idx.unique()]

    # split into train/test
    n_base_train = len(base_flat_idxs) // 2
    n_inflected_train = len(inflected_flat_idxs) // 2

    train_base_flat_idxs = base_flat_idxs[:n_base_train]
    train_inflected_flat_idxs = inflected_flat_idxs[:n_inflected_train]
    test_base_flat_idxs = base_flat_idxs[n_base_train:]
    test_inflected_flat_idxs = inflected_flat_idxs[n_inflected_train:]

    # now subsample/supersample to have matched number of instances
    n = min(max_num_vector_samples, max(len(train_base_flat_idxs), len(train_inflected_flat_idxs)))

    if len(train_base_flat_idxs) < n:
        train_base_flat_idxs = np.random.choice(train_base_flat_idxs, n, replace=True)
    else:
        train_base_flat_idxs = np.random.choice(train_base_flat_idxs, n, replace=False)
    if len(train_inflected_flat_idxs) < n:
        train_inflected_flat_idxs = np.random.choice(train_inflected_flat_idxs, n, replace=True)
    else:
        train_inflected_flat_idxs = np.random.choice(train_inflected_flat_idxs, n, replace=False)

    # match sample size on test base side
    if len(test_base_flat_idxs) < n:
        test_base_flat_idxs = np.random.choice(test_base_flat_idxs, n, replace=True)
    else:
        test_base_flat_idxs = np.random.choice(test_base_flat_idxs, n, replace=False)
    # don't need to worry about inflected sample size -- these are only used as a set of
    # indices for evaluation, so don't ever need to supersample

    assert set(train_base_flat_idxs).isdisjoint(set(train_inflected_flat_idxs))
    assert set(test_base_flat_idxs).isdisjoint(set(test_inflected_flat_idxs))

    assert set(train_base_flat_idxs).isdisjoint(set(test_base_flat_idxs))
    assert set(train_inflected_flat_idxs).isdisjoint(set(test_inflected_flat_idxs))

    train_base_flat_idxs = torch.tensor(train_base_flat_idxs).to(device)
    train_inflected_flat_idxs = torch.tensor(train_inflected_flat_idxs).to(device)
    test_base_flat_idxs = torch.tensor(test_base_flat_idxs).to(device)
    test_inflected_flat_idxs = torch.tensor(test_inflected_flat_idxs).to(device)

    with torch.no_grad():
        pair_difference = agg_gpu[train_inflected_flat_idxs] - agg_gpu[train_base_flat_idxs]
        pair_base = agg_gpu[test_base_flat_idxs]
        pair_predicted = pair_base + pair_difference
        pair_predicted /= torch.norm(pair_predicted, dim=1, keepdim=True)

        dists = 1 - nxn_cos_sim(pair_predicted, agg_gpu)
        # mean over instances of pair
        dists = dists.mean(0)

    # exclude the training items from distance evaluation
    dists[train_base_flat_idxs] = np.inf
    dists[train_inflected_flat_idxs] = np.inf

    sorted_indices = dists.argsort()
    ranks = torch.zeros_like(sorted_indices)
    ranks[sorted_indices] = torch.arange(len(sorted_indices)).to(sorted_indices.device)

    gt_rank = ranks[test_inflected_flat_idxs].min().item()
    gt_distance = dists[test_inflected_flat_idxs].min().item()

    nearest_neighbor = agg_src_gpu[sorted_indices[0]]

    return {
        "base_from": base,
        "inflected_from": inflected,
        "base_to": base,
        "inflected_to": inflected,

        "inflection": inflection,

        "predicted_label_idx": nearest_neighbor[0].item(),
        "predicted_label": state_space_spec.labels[nearest_neighbor[0].item()],
        "predicted_instance_idx": nearest_neighbor[1].item(),

        "gt_label": inflected,
        "gt_label_idx": inflected_idx,
        "gt_rank": gt_rank,
        "gt_distance": gt_distance,
        "num_instances": n,
    }

In [ ]:
study_instances = all_cross_instances[all_cross_instances.inflection.isin(("VBZ", "NNS")) & (all_cross_instances.exclude_main == False) & (all_cross_instances.is_regular == True) & (all_cross_instances.base_ambig_NN_VB == False)]

In [ ]:
all_results = []

for inflection, base, inflected in tqdm(study_instances[["inflection", "base", "inflected"]].value_counts().index, desc="Analogy trials"):
    try:
        result = run_same_word_trial(base, inflected, inflection)
        all_results.append(result)
    except Exception as e:
        print(f"Error processing {base}, {inflected}: {e}")

In [ ]:
all_results_df = pd.DataFrame(all_results)

### Save

In [ ]:
all_results_df.to_csv(f"{output_dir}/same_word_control_results.csv", index=False)